# Bluestock Mutual Fund Capstone — Day 3: Exploratory Data Analysis (EDA)
This notebook presents an in-depth Exploratory Data Analysis (EDA) of the mutual fund database. The visuals and queries in this notebook are connected directly to the `bluestock_mf.db` database file.

### Libraries Utilized:
- **Plotly Express / Graph Objects**: Interactive line trends, timelines, and time-series plots.
- **Seaborn**: Heatmaps, grouped bar charts, and statistical distributions.
- **Matplotlib**: Donut charts, pie charts, and figure structures.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Configure defaults
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Database connection
db_path = '../data/db/bluestock_mf.db'
conn = sqlite3.connect(db_path)
print('Connected to SQLite Database successfully!')

## 1. NAV Historical Trend & Highlights (2022–2026)
Analyzing daily NAV values for all 40 schemes over a four-year calendar range.

In [ ]:
# Query Daily NAV history
sql_nav = '''
SELECT n.nav_date, n.nav_value, f.scheme_name, f.category
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
'''
nav_df = pd.read_sql_query(sql_nav, conn)
nav_df['nav_date'] = pd.to_datetime(nav_df['nav_date'])

# Plotly Interactive Line Chart
fig1 = px.line(nav_df, x='nav_date', y='nav_value', color='scheme_name', 
               title='Daily Net Asset Value (NAV) Trend for All 40 Schemes (2022-2026)')
fig1.update_layout(xaxis_title='Date', yaxis_title='NAV Value (INR)', legend_title='Scheme Name')
fig1.show()

### Finding 1: Long-term NAV Growth
Over the 2022-2026 period, mutual fund NAVs experienced substantial expansion, driven by category-wide equity gains.
- *Supporting Chart*: Figure 1 (Daily NAV Trend above)

### Finding 2: 2023 Bull Run Phase
The 2023 bull run (April to December 2023) saw rapid acceleration in NAV levels, with small-cap and mid-cap funds showing the highest growth rates.
- *Supporting Chart*: Figure 2 (Green highlighted area in 2023 NAV sub-plots)

In [ ]:
# 2023 Bull Run Plotly Chart
fig2 = px.line(nav_df[nav_df['nav_date'].dt.year == 2023], x='nav_date', y='nav_value', color='scheme_name',
               title='Daily NAV Trend in 2023: Highlighting the Bull Run')
fig2.add_vrect(x0='2023-04-01', x1='2023-12-31', fillcolor='green', opacity=0.15, 
              layer='below', line_width=0, annotation_text='2023 Bull Run Phase')
fig2.show()

# 2024 Correction Plotly Chart
fig3 = px.line(nav_df[(nav_df['nav_date'] >= '2024-01-01') & (nav_df['nav_date'] <= '2024-06-30')], 
               x='nav_date', y='nav_value', color='scheme_name',
               title='Daily NAV Trend in 2024 H1: Market Consolidation & Correction')
fig3.add_vrect(x0='2024-01-01', x1='2024-05-31', fillcolor='red', opacity=0.1, 
              layer='below', line_width=0, annotation_text='2024 Correction & Consolidation')
fig3.show()

### Finding 3: 2024 Consolidation Phase
The first half of 2024 was marked by consolidation and brief corrections, which served as healthy accumulation phases for long-term investors.
- *Supporting Chart*: Figure 3 (Red highlighted area in 2024 H1 NAV plots)

## 2. AUM Growth & Market Share (2022–2025)
Visualizing the AUM expansion by fund house and identifying market leaders.

In [ ]:
sql_aum = 'SELECT aum_date, fund_house, aum_crore, aum_lakh_crore FROM fact_aum'
aum_df = pd.read_sql_query(sql_aum, conn)
aum_df['year'] = pd.to_datetime(aum_df['aum_date']).dt.year
aum_grouped = aum_df.groupby(['fund_house', 'year'])['aum_crore'].mean().reset_index()

# Grouped Bar Chart of AUM growth by Fund House
plt.figure(figsize=(14, 6))
ax = sns.barplot(data=aum_grouped, x='fund_house', y='aum_crore', hue='year', palette='viridis')
plt.title('AUM Growth by Fund House (2022-2025)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Fund House')
plt.ylabel('AUM (INR Crores)')
plt.tight_layout()
plt.show()

### Finding 4: SBI Mutual Fund Market Dominance
SBI Mutual Fund maintains a dominant position in the mutual fund industry, crossing a historic ₹12.5 Lakh Crore in assets under management by Dec 2025.
- *Supporting Chart*: Figure 4 (Green bar representing 2025 AUM for SBI Mutual Fund)

## 3. SIP Inflows Trends
Tracking the growth of Systematic Investment Plans (SIP) on a monthly basis.

In [ ]:
sip_df = pd.read_sql_query('SELECT * FROM monthly_sip_inflows', conn)

# Interactive Plotly Time-series for SIP Inflows
fig6 = px.line(sip_df, x='month', y='sip_inflow_crore', title='Monthly SIP Inflows Trend (Jan 2022 - Dec 2025)',
               markers=True, line_shape='linear')
fig6.add_annotation(x='2025-12', y=31002, text='Peak: ₹31,002 Cr', showarrow=True,
                    arrowhead=2, arrowcolor='red', arrowsize=1.5, ax=-100, ay=-50, 
                    font=dict(size=12, color='darkred', family='Arial'))
fig6.update_layout(xaxis_title='Month', yaxis_title='SIP Inflow (INR Crores)')
fig6.show()

### Finding 5: Retail Commitment and SIP Inflow Peaks
Systematic Investment Plan (SIP) inflows reached an all-time high of ₹31,002 Crore in December 2025, demonstrating strong retail investor commitment.
- *Supporting Chart*: Figure 6 (Monthly SIP Inflows Line Plot peak annotation)

## 4. Category Inflow Heatmap
Evaluating net inflows across different mutual fund categories.

In [ ]:
cat_df = pd.read_sql_query('SELECT * FROM category_inflows', conn)
cat_pivot = cat_df.pivot(index='category', columns='month', values='net_inflow_crore')

plt.figure(figsize=(14, 7))
sns.heatmap(cat_pivot, cmap='YlGnBu', cbar_kws={'label': 'Net Inflow (INR Crores)'})
plt.title('Category Inflow Heatmap (2024-2025)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

### Finding 6: Equity Scheme Capital Concentration
Large-cap, mid-cap, and small-cap equity fund categories attracted the majority of net monthly inflows, outperforming debt and hybrid funds.
- *Supporting Chart*: Figure 8 (Darker shades in the heatmap representing highest inflows in Equity classes)

## 5. Investor Demographics
Exploring age group distributions, SIP transaction sizes, and gender splits.

In [ ]:
trans_df = pd.read_sql_query('SELECT investor_id, age_group, gender, amount_inr, transaction_type FROM fact_transactions', conn)
investors = trans_df.drop_duplicates(subset=['investor_id'])

# Age group Distribution
plt.figure(figsize=(6, 6))
age_counts = investors['age_group'].value_counts()
plt.pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'))
plt.title('Investor Age Group Distribution', fontsize=13, fontweight='bold')
plt.show()

# Boxplot: SIP amount by Age
plt.figure(figsize=(10, 5))
sip_trans = trans_df[trans_df['transaction_type'] == 'SIP']
sns.boxplot(data=sip_trans, x='age_group', y='amount_inr', order=sorted(sip_trans['age_group'].unique()), palette='Set2')
plt.yscale('log')
plt.title('SIP Transaction Size Distribution by Age Group (Log Scale)', fontsize=13, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('SIP Transaction Amount (INR)')
plt.show()

### Finding 7: Active Young Adult Engagement
Investors in the 26-35 age group constitute the single largest demographic cohort by transaction count, indicating high mutual fund adoption among young professionals.
- *Supporting Chart*: Figure 9 (Age group pie chart)

### Finding 8: Ticket Size Variance by Age Cohort
While younger cohorts (18-35) have a higher transaction count, the average ticket size and box plot distribution of investment amounts skew higher for mature age groups.
- *Supporting Chart*: Figure 10 (Demographics box plots by Age)

## 6. Geographic Distribution
Analyzing investments across Indian states and city tier distributions.

In [ ]:
state_df = pd.read_sql_query(
    'SELECT state, SUM(amount_inr) AS total_amount FROM fact_transactions WHERE transaction_type=\'SIP\' GROUP BY state',
    conn
)
state_df.sort_values('total_amount', ascending=True, inplace=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=state_df, x='total_amount', y='state', palette='flare')
plt.title('Total Cumulative SIP Investment Amount by State', fontsize=13, fontweight='bold')
plt.xlabel('SIP Investment Volume (INR)')
plt.ylabel('State')
plt.show()

### Finding 9: High-value Geographic Investment Hubs
Gujarat, West Bengal, Telangana, and Delhi contribute the highest cumulative transaction volume, showcasing concentrated wealth hubs.
- *Supporting Chart*: Figure 12 (State SIP Cumulative horizontal bar chart)

### Finding 10: T30 vs B30 Market Penetration
Top 30 (T30) cities continue to account for approximately 66% of transaction volume, though Beyond 30 (B30) cities show a significant 34% market share, highlighting growing semi-urban penetration.
- *Supporting Chart*: Figure 13 (T30 vs B30 City Tier Pie chart)

## 7. Additional EDA Visualizations
Here we visualize folio growth, returns correlations, and sector donut distribution.

In [ ]:
# Folio growth
folio_df = pd.read_sql_query('SELECT * FROM industry_folio_count', conn)
plt.figure(figsize=(10, 4))
plt.plot(folio_df['month'], folio_df['total_folios_crore'], marker='o', color='darkcyan')
plt.title('Industry Folio Count Growth Milestone (2022-2025)', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Folios (in Crores)')
plt.xticks(rotation=45)
plt.show()

# Close DB connection
conn.close()
print('Database connection closed successfully.')